# 05 - 文件压缩：让数据"瘦身"

**Cambridge International AS & A Level Computer Science (9618) - Section 1.3**

---

> "一张 4K 照片 25MB，一首歌 30MB，一分钟视频 10GB——如果不压缩，你的硬盘一天就满了。"

本节深入讲解：
- 为什么需要压缩
- 无损 vs 有损压缩的原理
- RLE 行程编码（手把手教）
- JPEG 图像压缩原理
- MP3 音频压缩原理
- 通用文件减小方法

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'simhei' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = 'C:/Windows/Fonts/simhei.ttf'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import sys
try:
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Libraries loaded!')
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib', 'numpy', '-q'])
    import matplotlib.patches as patches
    from matplotlib.colors import ListedColormap
    import numpy as np
    from IPython.display import display, HTML, Image
    print('Installed & loaded!')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (10,6),
    'font.sans-serif': ['WenQuanYi Zen Hei','Noto Sans CJK SC','DejaVu Sans'], 'axes.unicode_minus': False})


---
## 1. 为什么需要文件压缩？

让我们算一笔账：

| 文件类型 | 未压缩大小 | 压缩后大小 | 压缩比 |
|:---|---:|---:|:---:|
| 1 张 4K 照片 (24-bit) | 25 MB | 3 MB (JPEG) | 8:1 |
| 1 首 3 分钟歌曲 (CD) | 30 MB | 3 MB (MP3) | 10:1 |
| 1 分钟 FHD 视频 | 9 GB | 150 MB (H.264) | 60:1 |
| 1 部 2 小时 4K 电影 | 3 TB | 8 GB (H.265) | 375:1 |

**没有压缩技术，现代互联网根本无法运行！**

- 一个 16GB 手机只能存 640 张未压缩照片（vs 5000+ 张 JPEG）
- 流媒体视频需要 1.2 Gbps 带宽（vs 实际用 5-25 Mbps）
- 一首歌下载要 30 秒（vs MP3 只要 3 秒）

---
## 2. 两种压缩思路

### 2.1 无损压缩 (Lossless Compression)

**核心思想：** 找到数据中的 **重复模式**，用更短的方式表达，不丢失任何信息。

就像把 "aaaaabbbbb" 写成 "5a5b"——信息完全一样，但更短。

**解压后能 100% 恢复原文件。**

**适用场景：**
- 文本文件（少一个字母都不行）
- 程序代码
- 电子表格
- 医学影像（不能丢失细节）

**常见格式：** ZIP, PNG, FLAC, 7z, RAR

### 2.2 有损压缩 (Lossy Compression)

**核心思想：** 去掉人类 **感知不到** 的信息，大幅减小文件。

就像把一首交响乐中你听不到的超高频和被掩盖的微弱声音删掉——你根本听不出区别。

**解压后无法恢复原文件。** 但人类几乎察觉不到差异。

**适用场景：**
- 照片（JPEG）
- 音乐（MP3, AAC）
- 视频（MP4, H.264, H.265）
- 流媒体

**常见格式：** JPEG, MP3, MP4, AAC, WebP

In [ ]:
# 无损 vs 有损 流程图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, title, color, fc, ec, sizes, lost, examples in [
    (ax1, 'LOSSLESS', 'green',
     ['#4CAF50','#81C784','#4CAF50'], '#2E7D32',
     [(0.3,4.5,2.8,1.5),(4.2,4.7,1.8,1.1),(7.2,4.5,2.8,1.5)], False,
     'ZIP, PNG, FLAC, RLE'),
    (ax2, 'LOSSY', '#E65100',
     ['#FF9800','#FFB74D','#FFCC80'], '#E65100',
     [(0.3,4.5,2.8,1.5),(4.6,4.9,1.2,0.7),(7.2,4.65,2.6,1.2)], True,
     'JPEG, MP3, MP4')]:

    ax.set_xlim(0, 10.5); ax.set_ylim(0, 7.5)
    ax.set_title(f'{title} Compression', fontsize=18, fontweight='bold', color=color)

    labels = ['Original\n100 KB',
              'Compressed\n' + ('60 KB' if not lost else '10 KB!'),
              'Restored\n' + ('100 KB' if not lost else '~95 KB')]

    for (x,y,w,h), c, lbl in zip(sizes, fc, labels):
        ax.add_patch(patches.FancyBboxPatch((x,y), w, h,
            boxstyle='round,pad=0.15', facecolor=c, edgecolor=ec, linewidth=2.5))
        tc = 'white' if c == fc[0] else 'black'
        ax.text(x+w/2, y+h/2, lbl, ha='center', va='center', fontsize=11, fontweight='bold', color=tc)

    ax.annotate('Compress', xy=(4.0, 5.25), xytext=(3.3, 5.25),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#333'),
                fontsize=10, ha='center', va='bottom')
    ax.annotate('Decompress', xy=(7.0, 5.25), xytext=(6.1, 5.25),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#333'),
                fontsize=10, ha='center', va='bottom')

    if not lost:
        msg = 'Original = Restored\n(IDENTICAL, zero data loss)'
        bg = '#E8F5E9'
    else:
        msg = 'Original != Restored\n(Some data permanently removed)'
        bg = '#FFF3E0'

    ax.text(5.25, 2, msg, ha='center', fontsize=13, fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.5', facecolor=bg, edgecolor=color, linewidth=2))
    ax.text(5.25, 0.5, f'Examples: {examples}', ha='center', fontsize=12, style='italic')
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. RLE 行程编码 (Run-Length Encoding)

### 3.1 基本原理

RLE 是最简单的 **无损** 压缩方法。

**核心思想：** 把连续重复的数据替换为 "重复次数 + 数据值"。

### 3.2 文本示例（逐步讲解）

原始字符串：`aaaaabbbbccddddd`

分析连续重复的部分：
```
a a a a a  ->  重复 5 次 'a'  ->  编码为 (5, 'a')
b b b b    ->  重复 4 次 'b'  ->  编码为 (4, 'b')
c c        ->  重复 2 次 'c'  ->  编码为 (2, 'c')
d d d d d  ->  重复 5 次 'd'  ->  编码为 (5, 'd')
```

RLE 结果：`5a 4b 2c 5d`

**大小对比：**
- 原始：16 个字符 = 16 bytes
- RLE：8 个值 = 8 bytes（每对 count+value 各占 1 byte）
- **压缩了 50%！**

### 3.3 什么时候 RLE 不好用？

字符串 `abcdabcd`：
```
a -> (1, 'a')
b -> (1, 'b')
c -> (1, 'c')
d -> (1, 'd')
a -> (1, 'a')
...
```
RLE 结果：`1a 1b 1c 1d 1a 1b 1c 1d` = 16 bytes

原始只有 8 bytes！**RLE 反而变大了！**

**结论：RLE 只对有大量连续重复的数据有效。**

In [ ]:
# RLE 编码器（带详细步骤）
def rle_encode(data):
    print(f'Original: "{data}"')
    print(f'Original size: {len(data)} bytes')
    print()
    print('Encoding process:')

    encoded = []
    i = 0
    while i < len(data):
        count = 1
        while i + count < len(data) and data[i + count] == data[i]:
            count += 1
        encoded.append((count, data[i]))
        run_str = data[i] * count
        print(f'  "{run_str}" -> run of {count} x \'{data[i]}\' -> ({count}, \'{data[i]}\')')
        i += count

    rle_str = ' '.join(f'{c}{v}' for c, v in encoded)
    rle_size = len(encoded) * 2

    print(f'\nRLE result: {rle_str}')
    print(f'Compressed size: {rle_size} bytes')

    ratio = rle_size / len(data) * 100
    saved = 100 - ratio
    print(f'Compression: {ratio:.0f}% of original (saved {saved:.0f}%)')

    if saved < 0:
        print('  WARNING: RLE made the data LARGER! Not suitable for this data.')
    elif saved == 0:
        print('  No compression achieved.')
    else:
        print(f'  Reduced from {len(data)} to {rle_size} bytes.')
    return encoded

print('='*55)
print('Example 1: Good case for RLE')
print('='*55)
rle_encode('aaaaabbbbccddddd')

print('\n' + '='*55)
print('Example 2: Another good case')
print('='*55)
rle_encode('AAAABBCCCCDDDDDDEE')

print('\n' + '='*55)
print('Example 3: BAD case for RLE')
print('='*55)
rle_encode('abcabcabc')

In [ ]:
# RLE on a black & white image
letter_F = np.array([
    [1,1,1,1,1,1,1,1],[1,0,0,0,0,0,0,1],[1,0,1,1,1,1,1,1],[1,0,1,1,1,1,1,1],
    [1,0,0,0,0,0,1,1],[1,0,1,1,1,1,1,1],[1,0,1,1,1,1,1,1],[1,0,1,1,1,1,1,1]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cmap_bw = ListedColormap(['#333', 'white'])
ax1.imshow(letter_F, cmap=cmap_bw, interpolation='nearest')
ax1.set_title('Letter "F" (8x8 grid)', fontsize=14, fontweight='bold')
for i in range(8):
    for j in range(8):
        c = 'yellow' if letter_F[i,j]==0 else 'gray'
        ax1.text(j, i, str(letter_F[i,j]), ha='center', va='center',
                fontsize=12, fontweight='bold', color=c)
ax1.set_xticks(np.arange(-0.5, 8, 1), minor=True)
ax1.set_yticks(np.arange(-0.5, 8, 1), minor=True)
ax1.grid(which='minor', color='gray', linewidth=1)

# RLE encode
flat = letter_F.flatten()
pairs = []; i = 0
while i < len(flat):
    c = 1
    while i+c < len(flat) and flat[i+c] == flat[i]: c += 1
    pairs.append(f"{c}{'W' if flat[i]==1 else 'B'}")
    i += c

txt = 'RLE Encoding:\n\n' + ' '.join(pairs)
txt += f'\n\nOriginal: 64 bytes (8x8 grid)'
txt += f'\nCompressed: {len(pairs)*2} bytes ({len(pairs)} pairs x 2)'
txt += f'\nSaved: {(1 - len(pairs)*2/64)*100:.0f}%'

ax2.text(0.1, 0.5, txt, fontsize=12, fontfamily='monospace', va='center',
         bbox=dict(boxstyle='round,pad=1', fc='lightyellow', ec='orange', lw=2))
ax2.axis('off')
ax2.set_title('RLE Compression Result', fontsize=14, fontweight='bold')

plt.tight_layout(); plt.show()

---
## 4. MP3 压缩原理

MP3 使用 **有损** 压缩，核心技术叫 **感知音乐整形 (Perceptual Music Shaping)**。

### 4.1 人耳的"缺陷"

MP3 利用了人耳的两个特性：

**1. 频率范围有限**
- 人耳只能听到 20 Hz ~ 20,000 Hz
- 低于 20 Hz 和高于 20,000 Hz 的声音可以直接删除
- 你根本听不到，删了也不影响体验

**2. 听觉掩蔽效应 (Auditory Masking)**
- 当一个响亮的声音和一个微弱的声音同时播放时
- 人耳只能听到响的那个，轻的被"掩盖"了
- 所以可以把轻的声音也删掉

### 4.2 压缩效果

通过这两个技巧，MP3 可以去掉约 **90%** 的数据！

| 质量 | 比特率 | 3分钟歌曲大小 |
|:---|:---|:---|
| 原始 CD | 1,411 kbps | ~30 MB |
| MP3 高质量 | 320 kbps | ~7 MB |
| MP3 标准 | 192 kbps | ~4 MB |
| MP3 低质量 | 128 kbps | ~3 MB |
| MP3 最低 | 64 kbps | ~1.5 MB |

**比特率 (Bit Rate)** = 每秒使用多少 bit 来存储音频
- 比特率越高 = 质量越好 = 文件越大

In [ ]:
# MP3 感知音乐整形可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
t = np.linspace(0, 0.01, 1000)

# What we hear
audible = np.sin(2*np.pi*440*t)  # 440 Hz A note
# What gets removed
quiet = 0.1 * np.sin(2*np.pi*445*t)  # Masked by 440Hz
ultra = 0.3 * np.sin(2*np.pi*25000*t)  # Above hearing range

axes[0].plot(t*1000, audible + quiet + ultra, 'b-', lw=1)
axes[0].set_title('Original Sound\n(All frequencies)', fontsize=13, fontweight='bold')
axes[0].set_ylim(-2, 2); axes[0].grid(True, alpha=0.3)

axes[1].plot(t*1000, quiet, 'r-', lw=1, label='Masked sound (too quiet)')
axes[1].plot(t*1000, ultra, 'orange', lw=1, label='Ultrasonic (>20kHz)')
axes[1].set_title('Removed by MP3\n(You cannot hear these!)', fontsize=13, fontweight='bold', color='red')
axes[1].legend(fontsize=9); axes[1].set_ylim(-2, 2); axes[1].grid(True, alpha=0.3)

axes[2].plot(t*1000, audible, 'g-', lw=1)
axes[2].set_title('MP3 Result\n(Only audible sound)', fontsize=13, fontweight='bold', color='green')
axes[2].set_ylim(-2, 2); axes[2].grid(True, alpha=0.3)

for ax in axes: ax.set_xlabel('Time (ms)')
plt.suptitle('MP3: Perceptual Music Shaping', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

cd = 44100*16*2*180/8/1e6
print(f'\n3-minute song comparison:')
print(f'  WAV (uncompressed): {cd:.1f} MB')
print(f'  MP3 (128 kbps):     {128*180/8/1000:.1f} MB')
print(f'  MP3 (320 kbps):     {320*180/8/1000:.1f} MB')
print(f'  Compression ratio:  ~{(1 - 128*180/8/1000/cd)*100:.0f}% reduction at 128kbps')

---
## 5. JPEG 压缩原理

JPEG 也是 **有损** 压缩，利用了人眼的特性：

### 人眼对亮度比对颜色更敏感

JPEG 的做法：
1. **颜色空间转换**：RGB -> YCbCr（亮度 + 色度）
2. **降采样色度**：人眼对颜色细节不敏感，可以降低色度分辨率
3. **分块处理**：将图像分成 8x8 的小块
4. **DCT 变换**：用数学方法找到每个块中的频率成分
5. **量化**：去掉高频细节（人眼不敏感的部分）
6. **编码**：用无损压缩进一步缩小

典型效果：文件缩小 **5-15 倍**，人眼几乎看不出区别。

### JPEG vs PNG

| | JPEG | PNG |
|:---|:---|:---|
| 压缩类型 | 有损 | 无损 |
| 透明度 | 不支持 | 支持 (Alpha) |
| 最佳用途 | 照片 | 图标、截图、文字图 |
| 文件大小 | 很小 | 较大 |
| 重复压缩 | 越压越差 | 不会降质 |

---
## 6. 通用文件减小方法

除了压缩算法，还有一些通用方法可以减小文件：

| 文件类型 | 方法 | 效果 |
|:---|:---|:---|
| **图像** | 降低分辨率 | 像素数减半 -> 文件减半 |
| **图像** | 降低色彩深度 | 24位->8位 = 文件变为1/3 |
| **图像** | 裁剪 (Crop) | 去掉不需要的部分 |
| **声音** | 降低采样率 | 44.1kHz->22kHz = 文件减半 |
| **声音** | 降低位深度 | 16位->8位 = 文件减半 |
| **声音** | 单声道代替立体声 | 文件减半 |
| **视频** | 降低帧率 | 60fps->30fps = 文件减半 |
| **视频** | 降低分辨率 | 4K->1080p = 文件变为1/4 |

**注意：** 这些方法都会降低质量！需要在质量和文件大小之间找平衡。

---
## 7. 综合练习

1. 解释无损压缩和有损压缩的区别，各举两个例子。
2. 对字符串 `AAAABBCCCCDDDDDDEE` 进行 RLE 编码，计算压缩率。
3. 为什么 RLE 对 `ABCDABCDABCD` 无效？
4. 解释 MP3 压缩的两个核心原理。
5. 比特率 128kbps 和 320kbps 的 MP3 有什么区别？
6. 为什么 JPEG 适合照片但不适合文字截图？（提示：想想锐利边缘）
7. 一个音频文件太大了，列出 4 种减小它的方法（2种压缩 + 2种非压缩）。
8. 为什么医学影像必须用无损压缩而不是有损压缩？

In [ ]:
# Answers
print('Q1: Lossless = original fully recoverable (ZIP, PNG, FLAC, RLE)')
print('    Lossy = some data permanently lost (JPEG, MP3, MP4, AAC)')

print('\nQ2:')
rle_encode('AAAABBCCCCDDDDDDEE')

print("""
Q3: ABCDABCDABCD has NO consecutive repeated characters.
    RLE produces: 1A 1B 1C 1D 1A 1B 1C 1D 1A 1B 1C 1D
    That is 24 bytes vs original 12 bytes - TWICE as large!
    RLE only works when there are long runs of repeated data.

Q4: MP3 uses two principles:
    1. Frequency filtering: removes sounds outside 20Hz-20kHz (inaudible)
    2. Auditory masking: when a loud and quiet sound play together,
       removes the quiet one (brain can't hear it anyway)

Q5: 128kbps uses 128,000 bits per second of audio.
    320kbps uses 320,000 bits per second.
    Higher bitrate = more data preserved = better quality = larger file.
    128kbps: acceptable for casual listening
    320kbps: nearly indistinguishable from CD quality

Q6: JPEG works by removing high-frequency detail (sharp edges).
    Photos have smooth gradients -> JPEG works great.
    Text has very sharp edges -> JPEG creates visible artifacts
    around letters (smudging/ringing). PNG preserves sharp edges perfectly.

Q7: 4 ways to reduce audio file size:
    Compression: 1. Use MP3 lossy compression
                 2. Use FLAC lossless compression
    Non-compression: 3. Reduce sampling rate
                     4. Reduce bit depth (or use mono)

Q8: Medical images (X-rays, MRIs) may contain tiny details
    that are critical for diagnosis. Lossy compression might
    remove a small shadow that indicates a tumour.
    Doctors need to see EVERYTHING in the image.
    Even a tiny loss of detail could be life-threatening.""")

---
## 全章总结

| 主题 | 核心知识点 |
|:---|:---|
| **像素** | 图像最小单元，每像素存一个颜色 |
| **色彩深度** | n bits = 2^n 种颜色 (二进制指数增长) |
| **RGB** | 8+8+8=24位真彩色 = 16.7M 种颜色 |
| **图像大小** | = W x H x depth / 8 (bytes) |
| **PPI** | = 对角线像素 / 屏幕英寸 |
| **矢量图** | 数学描述；缩放不失真；文件小 |
| **位图** | 像素网格；真实感强；文件大 |
| **采样率** | 每秒采样数(Hz)；越高越好 |
| **位深度** | 每样本位数；越高越精确 |
| **声音大小** | = rate x bits x channels x seconds / 8 |
| **无损压缩** | 完全可恢复 (RLE, ZIP, PNG) |
| **有损压缩** | 不可恢复但更小 (JPEG, MP3, MP4) |
| **RLE** | 连续重复 -> 次数+值 |
| **MP3** | 去除听不到的声音（感知整形） |
| **JPEG** | 去除看不到的颜色细节 |

---
**Congratulations!** You have completed Section 1.2 & 1.3!

**Next chapter:** Chapter 2 - Communication & Networking